<a href="https://colab.research.google.com/github/Dacon-TeamPK/dacon-physics-vision-ai/blob/main/%EA%B5%AC%EC%A1%B0%EB%AC%BC_%EC%95%88%EC%A0%95%EC%84%B1_%EB%AC%BC%EB%A6%AC_%EC%B6%94%EB%A1%A0_AI_%EA%B2%BD%EC%A7%84%EB%8C%80%ED%9A%8C_MultiviewConvNeXt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.chdir('/content/drive/MyDrive')

In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np

In [4]:
print(os.getcwd())

/content/drive/MyDrive


In [5]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import ConvNeXt_Tiny_Weights
from tqdm.auto import tqdm

# =========================
# 하이퍼파라미터 설정
# =========================
CFG = {
    'IMG_SIZE': 224,
    'EPOCHS': 5,
    'LEARNING_RATE': 2e-5,
    'BATCH_SIZE': 8,
    'SEED': 42,
    'PATIENCE': 2
}

# =========================
# Seed 고정
# =========================
def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG['SEED'])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("CFG:", CFG)
print("device:", device)

# =========================
# 데이터 로드
# =========================
train_df = pd.read_csv('/content/drive/MyDrive/colab/train.csv')
val_df = pd.read_csv('/content/drive/MyDrive/colab/dev.csv')
test_df = pd.read_csv('/content/drive/MyDrive/colab/sample_submission.csv')

print(f"학습 데이터 개수: {len(train_df)}")
print(f"검증 데이터 개수: {len(val_df)}")
print(f"테스트 데이터 개수: {len(test_df)}")

# =========================
# Dataset
# =========================
class MultiViewDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.label_map = {'stable': 0, 'unstable': 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        sample_id = str(self.df.iloc[idx]['id'])
        folder_path = os.path.join(self.root_dir, sample_id)

        views = []
        for name in ["front", "top"]:
            img_path = os.path.join(folder_path, f"{name}.png")

            if not os.path.exists(img_path):
                raise FileNotFoundError(f"파일이 존재하지 않습니다: {img_path}")

            image = Image.open(img_path).convert("RGB")

            if self.transform:
                image = self.transform(image)

            views.append(image)

        if self.is_test:
            return views, sample_id

        label = self.label_map[self.df.iloc[idx]['label']]
        label = torch.tensor(label, dtype=torch.float32)

        return views, label

# =========================
# Transform
# =========================
train_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.3,
        hue=0.05
    ),
    transforms.RandomAffine(
        degrees=7,
        translate=(0.05, 0.05),
        scale=(0.90, 1.10)
    ),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# =========================
# DataLoader
# =========================
train_dataset = MultiViewDataset(
    train_df,
    '/content/drive/MyDrive/colab/train',
    transform=train_transform,
    is_test=False
)

val_dataset = MultiViewDataset(
    val_df,
    '/content/drive/MyDrive/colab/dev',
    transform=test_transform,
    is_test=False
)

test_dataset = MultiViewDataset(
    test_df,
    '/content/drive/MyDrive/colab/test',
    transform=test_transform,
    is_test=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG['BATCH_SIZE'],
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG['BATCH_SIZE'],
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CFG['BATCH_SIZE'],
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# =========================
# 모델
# =========================
class MultiViewConvNeXt(nn.Module):
    def __init__(self, num_classes=1):
        super(MultiViewConvNeXt, self).__init__()

        self.backbone = models.convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT)
        self.feature_extractor = self.backbone.features
        self.avgpool = self.backbone.avgpool

        self.classifier = nn.Sequential(
            nn.LayerNorm(768 * 2),
            nn.Linear(768 * 2, 512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def extract_feature(self, x):
        x = self.feature_extractor(x)   # [B, 768, H, W]
        x = self.avgpool(x)             # [B, 768, 1, 1]
        x = torch.flatten(x, 1)         # [B, 768]
        return x

    def forward(self, views):
        f1 = self.extract_feature(views[0])
        f2 = self.extract_feature(views[1])

        combined = torch.cat((f1, f2), dim=1)
        return self.classifier(combined).view(-1)

# =========================
# Metric
# =========================
def competition_logloss(y_true, y_prob):
    y_true = np.array(y_true, dtype=np.float64)
    y_prob = np.array(y_prob, dtype=np.float64)

    eps = 1e-15
    y_prob = np.clip(y_prob, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))

# =========================
# Train / Validate
# =========================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    train_loss = 0.0

    for views, labels in tqdm(loader, desc="Training"):
        views = [v.to(device) for v in views]
        labels = labels.to(device).float()

        optimizer.zero_grad()
        outputs = model(views)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    return train_loss / len(loader)

def validate(model, loader, criterion, device):
    model.eval()
    val_loss = 0.0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for views, labels in tqdm(loader, desc="Validation"):
            views = [v.to(device) for v in views]
            labels = labels.to(device).float()

            outputs = model(views)
            loss = criterion(outputs, labels)
            probs = torch.sigmoid(outputs)

            val_loss += loss.item()
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(loader)
    logloss_score = competition_logloss(all_labels, all_probs)
    acc_score = np.mean((np.array(all_probs) > 0.5) == np.array(all_labels))

    return avg_val_loss, logloss_score, acc_score

# =========================
# Inference
# =========================
def inference(model, loader, device):
    model.eval()
    all_ids = []
    all_probs = []

    with torch.no_grad():
        for views, sample_ids in tqdm(loader, desc="Inference"):
            views = [v.to(device) for v in views]
            outputs = model(views)
            probs = torch.sigmoid(outputs)

            all_ids.extend(sample_ids)
            all_probs.extend(probs.cpu().numpy())

    return all_ids, all_probs

# =========================
# 학습
# =========================
model = MultiViewConvNeXt().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(
    model.parameters(),
    lr=CFG['LEARNING_RATE'],
    weight_decay=5e-4
)

best_logloss = float('inf')
patience_counter = 0
best_model_path = 'best_convnext_multiview.pth'

print(f"Start training for {CFG['EPOCHS']} epochs")

for epoch in range(1, CFG['EPOCHS'] + 1):
    avg_train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    avg_val_loss, val_logloss, val_acc = validate(model, val_loader, criterion, device)

    print(f"Epoch [{epoch}]")
    print(f"  - Train Loss    : {avg_train_loss:.4f}")
    print(f"  - Val Loss      : {avg_val_loss:.4f}")
    print(f"  - Val Log-Loss  : {val_logloss:.6f}")
    print(f"  - Val Acc       : {val_acc:.4f}")

    if val_logloss < best_logloss:
        best_logloss = val_logloss
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
        print(f"  -> Best model saved! (best logloss: {best_logloss:.6f})")
    else:
        patience_counter += 1
        print(f"  -> No improvement. Patience: {patience_counter}/{CFG['PATIENCE']}")

    if patience_counter >= CFG['PATIENCE']:
        print("Early stopping triggered.")
        break

print(f"\nBest Val Log-Loss: {best_logloss:.6f}")

# =========================
# Best model 로드 후 추론
# =========================
model.load_state_dict(torch.load(best_model_path, map_location=device))

test_ids, test_probs = inference(model, test_loader, device)

submission = pd.DataFrame({
    'id': test_ids,
    'unstable_prob': test_probs
})
submission['stable_prob'] = 1 - submission['unstable_prob']
submission = submission[['id', 'unstable_prob', 'stable_prob']]

submission.to_csv('submission_convnext.csv', index=False)
print("\nsubmission_convnext.csv 저장 완료")
print(submission.head())

CFG: {'IMG_SIZE': 224, 'EPOCHS': 5, 'LEARNING_RATE': 2e-05, 'BATCH_SIZE': 8, 'SEED': 42, 'PATIENCE': 2}
device: cuda
학습 데이터 개수: 1000
검증 데이터 개수: 100
테스트 데이터 개수: 1000
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 124MB/s]


Start training for 5 epochs


Training:   0%|          | 0/125 [00:00<?, ?it/s]

Validation:   0%|          | 0/13 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/13 [00:00<?, ?it/s]

Epoch [5]
  - Train Loss    : 0.0189
  - Val Loss      : 0.3746
  - Val Log-Loss  : 0.389076
  - Val Acc       : 0.8800
  -> No improvement. Patience: 1/2

Best Val Log-Loss: 0.338762


Inference:   0%|          | 0/125 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d12ced03d80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d12ced03d80>self._shutdown_workers()^^

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
Traceback (most recent call last):
      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
if w.is_alive():    
 self._shutdown_workers() 
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
       if w.is_alive(): 
^^ ^  ^ ^ ^ ^^ ^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^
^^  ^ ^ 
   File "/usr/lib


submission_convnext.csv 저장 완료
          id  unstable_prob  stable_prob
0  TEST_0001       0.003320     0.996680
1  TEST_0002       0.998104     0.001896
2  TEST_0003       0.997606     0.002394
3  TEST_0004       0.997127     0.002873
4  TEST_0005       0.935916     0.064084
